# Augmented Dataset Scrollie — Segmentation Viewer

Slice-by-slice viewer for all algorithms run on the **our_augmented_dataset** (20 volumes).

- **Images**: `our_augmented_dataset/{stem}_augmented000_water.nii.gz`
- **GT**: `our_augmented_dataset/{stem}_augmented000_seg.nii.gz` (8 labels — 4 bilateral muscle groups)

**How to use:**
1. Run all cells.
2. Pick up to **two algorithms** from the dropdowns.
3. Pick a sample from the Stack dropdown.
4. Toggle **Show GT** for ground-truth overlay.
5. Drag the slice slider.

In [1]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox
from IPython.display import display

In [2]:
# ── Label maps ─────────────────────────────────────────────────────────────────

# Ground truth: myosegmenTUM combined_gt scheme — 4 bilateral muscle groups
AUG_GT_LABELS = {
    1: ('L_Gracilis',   (0.12, 0.47, 0.71)),
    2: ('L_Hamstrings', (0.20, 0.63, 0.17)),
    3: ('L_Quadriceps', (0.89, 0.10, 0.11)),
    4: ('L_Sartorius',  (1.00, 0.50, 0.00)),
    5: ('R_Gracilis',   (0.60, 0.39, 0.64)),
    6: ('R_Hamstrings', (0.55, 0.34, 0.29)),
    7: ('R_Quadriceps', (0.89, 0.47, 0.76)),
    8: ('R_Sartorius',  (0.74, 0.74, 0.13)),
}

# MuscleMap WB — 7xxx scheme
MM_WB_LABELS = {
    7101: 'VastusLat_L',      7102: 'VastusLat_R',
    7111: 'VastusInt_L',      7112: 'VastusInt_R',
    7121: 'VastusMed_L',      7122: 'VastusMed_R',
    7131: 'RectusFem_L',      7132: 'RectusFem_R',
    7141: 'Sartorius_L',      7142: 'Sartorius_R',
    7151: 'Gracilis_L',       7152: 'Gracilis_R',
    7161: 'Semimem_L',        7162: 'Semimem_R',
    7171: 'Semiten_L',        7172: 'Semiten_R',
    7181: 'BicepsFemL_L',     7182: 'BicepsFemL_R',
    7201: 'AdductorMag_L',    7202: 'AdductorMag_R',
    7221: 'AdductorBrev_L',   7222: 'AdductorBrev_R',
}

# MuscleMap Thigh — 1–28 scheme
MM_THIGH_LABELS = {
    1:  'VastusLat_L',      2:  'VastusLat_R',
    3:  'VastusInt_L',      4:  'VastusInt_R',
    5:  'VastusMed_L',      6:  'VastusMed_R',
    7:  'RectusFem_L',      8:  'RectusFem_R',
    9:  'Sartorius_L',      10: 'Sartorius_R',
    11: 'Gracilis_L',       12: 'Gracilis_R',
    13: 'Semimem_L',        14: 'Semimem_R',
    15: 'Semiten_L',        16: 'Semiten_R',
    17: 'BicepsFemL_L',     18: 'BicepsFemL_R',
    19: 'BicepsFemS_L',     20: 'BicepsFemS_R',
    21: 'AddMag_L',         22: 'AddMag_R',
    23: 'AddLong_L',        24: 'AddLong_R',
    25: 'AddBrev_L',        26: 'AddBrev_R',
    27: 'Femur_L',          28: 'Femur_R',
}

HIRR_LABELS = {
    1:  'Sartorius',     2:  'RectusFem',
    3:  'VastusLat',     4:  'VastusInt',
    5:  'VastusMed',     6:  'AddMag',
    7:  'Gracilis',      8:  'BicepsFemL',
    9:  'Semiten',       10: 'Semimem',
    11: 'BicepsFemS',
}

MUSEG_LABELS = {
    1:  'VastusLat',    2:  'VastusInt',
    3:  'VastusMed',    4:  'RectusFem',
    5:  'Sartorius',    6:  'Gracilis',
    7:  'Semimem',      8:  'Semiten',
    9:  'BicepsFemL',   10: 'BicepsFemS',
    11: 'AddMag',       12: 'AddLong',
    13: 'AddBrev',
}

print('Label maps defined.')

Label maps defined.


In [3]:
# ── Configuration ───────────────────────────────────────────────────────────────
EVAL_DIR  = r'C:\Projects\dissector\eval_notebooks'
IMG_DIR   = os.path.join(EVAL_DIR, 'our_augmented_dataset')

ALGORITHMS = {
    'MuscleMap WB': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb', 'augmented_segs'),
        'glob':      '*_water_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MM_WB_LABELS,
    },
    'MuscleMap Thigh': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_thigh', 'augmented_segs'),
        'glob':      '*_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MM_THIGH_LABELS,
    },
    'Hirriririir': {
        'seg_dir':   os.path.join(EVAL_DIR, 'multimodal-multiethnic', 'augmented_segs'),
        'glob':      '*_thigh_seg.nii.gz',
        'fmt':       'nifti',
        'label_map': HIRR_LABELS,
    },
    'MuSeg': {
        'seg_dir':   os.path.join(EVAL_DIR, 'museg', 'augmented_segs'),
        'glob':      '*_dseg.nii.gz',
        'fmt':       'nifti',
        'label_map': MUSEG_LABELS,
    },
    'Dafne': {
        'seg_dir':   os.path.join(EVAL_DIR, 'dafne', 'augmented_segs'),
        'glob':      '*_dafne_thigh.npz',
        'fmt':       'npz',
        'label_map': None,
    },
    'MedCLIP-SAMv2': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medclipsamv2', 'augmented_segs'),
        'glob':      '*_medclipsamv2.npz',
        'fmt':       'npz',
        'label_map': None,
    },
    'MedCLIP-SAMv2 Text+Boxes': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medclipsamv2textboxes', 'augmented_segs'),
        'glob':      '*_mcsam2textboxes.npz',
        'fmt':       'npz',
        'label_map': None,
    },
    'MedSegDiff': {
        'seg_dir':   os.path.join(EVAL_DIR, 'medsegdiff', 'augmented_segs'),
        'glob':      '*_seg.npz',
        'fmt':       'npz',
        'label_map': None,
    },
    'SLM-SAM2': {
        'seg_dir':   os.path.join(EVAL_DIR, 'muscle_map_wb+slmsam', 'augmented_segs'),
        'glob':      '*_slmsam2.npz',
        'fmt':       'npz',
        'label_map': None,
    },
}

def _count(cfg):
    return len(glob.glob(os.path.join(cfg['seg_dir'], cfg['glob'])))

AVAILABLE = {
    name: cfg for name, cfg in ALGORITHMS.items()
    if os.path.isdir(cfg['seg_dir']) and _count(cfg) > 0
}

print(f'Available algorithms ({len(AVAILABLE)}/{len(ALGORITHMS)}):')
for name, cfg in AVAILABLE.items():
    print(f'  {name}: {_count(cfg)} files')

Available algorithms (4/9):
  MuscleMap WB: 9 files
  MuscleMap Thigh: 20 files
  Hirriririir: 20 files
  MuSeg: 20 files


In [4]:
# ── Helpers ─────────────────────────────────────────────────────────────────────

def stem_from_path(seg_path):
    """Extract stem (e.g. 'HV001_1_stack1') from any augmented seg filename."""
    base = os.path.basename(seg_path)
    m    = re.match(r'^(.+?)_augmented', base)
    return m.group(1) if m else base.split('_')[0]


def get_stacks(algo_name):
    """Return {stem: seg_path} dict sorted by stem."""
    cfg   = AVAILABLE[algo_name]
    files = glob.glob(os.path.join(cfg['seg_dir'], cfg['glob']))
    result = {}
    for f in files:
        stem = stem_from_path(f)
        result[stem] = f
    return dict(sorted(result.items()))


def load_image(stem):
    """Load water NIfTI → percentile-normalised (D,H,W) float32."""
    path = os.path.join(IMG_DIR, f'{stem}_augmented000_water.nii.gz')
    arr  = sitk.GetArrayFromImage(sitk.ReadImage(path)).astype(np.float32)
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def load_gt(stem):
    """Load ground-truth seg → (D,H,W,4) RGBA overlay + patches."""
    path    = os.path.join(IMG_DIR, f'{stem}_augmented000_seg.nii.gz')
    seg_arr = sitk.GetArrayFromImage(sitk.ReadImage(path)).astype(np.int32)
    alpha   = 0.5
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for label, (name, colour) in AUG_GT_LABELS.items():
        if not np.any(seg_arr == label):
            continue
        c = (*colour, alpha)
        rgba[seg_arr == label] = c
        patches.append(mpatches.Patch(color=colour, alpha=0.8, label=name))
    return rgba, patches


def build_nifti_overlay(seg_arr, label_map, alpha=0.5):
    present = {k: v for k, v in label_map.items() if np.any(seg_arr == k)}
    n       = max(len(present), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for i, (orig, name) in enumerate(present.items(), 1):
        c = (*cmap(i - 1)[:3], alpha)
        rgba[seg_arr == orig] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def build_npz_overlay(data, alpha=0.5):
    names = list(data.files)
    n     = max(len(names), 1)
    cmap  = plt.colormaps['tab20'].resampled(n)
    shape = data[names[0]].shape
    rgba  = np.zeros((*shape, 4), dtype=np.float32)
    patches = []
    for i, name in enumerate(names, 1):
        c = (*cmap(i - 1)[:3], alpha)
        rgba[data[name] > 0] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def load_seg_overlay(seg_path, cfg):
    if cfg['fmt'] == 'nifti':
        arr = sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).astype(np.int32)
        return build_nifti_overlay(arr, cfg['label_map'])
    else:
        return build_npz_overlay(np.load(seg_path))


print('Helpers ready.')

Helpers ready.


In [5]:
# ── Widgets ─────────────────────────────────────────────────────────────────────

if not AVAILABLE:
    print('No algorithms available yet — run the Lambda notebooks and download results.')
else:
    ALGO_OPTIONS = ['— none —'] + list(AVAILABLE)

    algo1_dd   = Dropdown(options=list(AVAILABLE), description='Algorithm 1:',
                          layout=widgets.Layout(width='420px'))
    algo2_dd   = Dropdown(options=ALGO_OPTIONS, value='— none —',
                          description='Algorithm 2:',
                          layout=widgets.Layout(width='420px'))
    stack_dd   = Dropdown(options=[], description='Stack:',
                          layout=widgets.Layout(width='280px'))
    slice_sl   = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                           layout=widgets.Layout(width='600px'))
    show_gt_cb = widgets.Checkbox(value=False, description='Show GT',
                                   indent=False, layout=widgets.Layout(width='110px'))
    out = widgets.Output()

    _cache    = {}
    _gt_cache = {}


    def _load(algo_name, stem):
        key = (algo_name, stem)
        if key not in _cache:
            cfg      = AVAILABLE[algo_name]
            stacks   = get_stacks(algo_name)
            seg_path = stacks[stem]
            img_norm = load_image(stem)
            overlay, patches = load_seg_overlay(seg_path, cfg)
            _cache[key] = (img_norm, overlay, patches)
        return _cache[key]


    def _load_gt(stem):
        if stem not in _gt_cache:
            _gt_cache[stem] = load_gt(stem)
        return _gt_cache[stem]


    def render(algo1, algo2, stem, slice_idx, show_gt):
        if not stem:
            return
        try:
            img_norm, ov1, patches1 = _load(algo1, stem)
        except Exception as e:
            with out:
                out.clear_output(wait=True)
                print(f'Error loading {algo1}: {e}')
            return

        img = img_norm[slice_idx]

        has_algo2 = algo2 != '— none —' and algo2 in AVAILABLE
        ov2, patches2 = None, []
        if has_algo2:
            stacks2 = get_stacks(algo2)
            if stem in stacks2:
                try:
                    _, ov2, patches2 = _load(algo2, stem)
                except Exception:
                    has_algo2 = False
            else:
                has_algo2 = False

        gt_ov, gt_patches, gt_err = None, [], None
        if show_gt:
            try:
                gt_ov, gt_patches = _load_gt(stem)
            except Exception as e:
                gt_err = str(e)

        panels = [('Water image', None, None)]
        if show_gt:
            panels.append(('Ground truth (8 muscles)',
                            gt_ov, gt_patches if gt_err is None else []))
        panels.append((algo1, ov1, patches1))
        if has_algo2:
            panels.append((algo2, ov2, patches2))
        else:
            lbl = 'Algorithm 2 — select above' if algo2 == '— none —' \
                  else f'{algo2}\n(no result for this stack)'
            panels.append((lbl, None, None))

        n_panels  = len(panels)
        fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
        if n_panels == 1:
            axes = [axes]

        for ax, (title, overlay, patches) in zip(axes, panels):
            ax.imshow(img, cmap='gray', origin='lower')
            if overlay is not None:
                ax.imshow(overlay[slice_idx], origin='lower')
            if patches:
                ax.legend(handles=patches, loc='lower right', fontsize=5,
                          framealpha=0.7, ncol=2)
            color = 'gray' if overlay is None and not patches else 'black'
            ax.set_title(title, fontsize=10, color=color)
            ax.axis('off')

        fig.suptitle(f'{stem}  —  slice {slice_idx}', fontsize=11)
        plt.tight_layout()
        with out:
            out.clear_output(wait=True)
            plt.show()
            if gt_err:
                print(f'[GT] {gt_err}')


    def _rerender(*_):
        render(algo1_dd.value, algo2_dd.value, stack_dd.value,
               slice_sl.value, show_gt_cb.value)


    def on_algo1_change(change):
        _cache.clear()
        stacks = get_stacks(change['new'])
        stack_dd.options = list(stacks)
        if stacks:
            stack_dd.value = list(stacks)[0]
            slice_sl.max   = _load(change['new'], stack_dd.value)[0].shape[0] - 1
            slice_sl.value = 0
        _rerender()


    def on_stack_change(change):
        if change['new']:
            img_norm, *_ = _load(algo1_dd.value, change['new'])
            slice_sl.max   = img_norm.shape[0] - 1
            slice_sl.value = 0
        _rerender()


    algo1_dd.observe(on_algo1_change, names='value')
    algo2_dd.observe(lambda _: _rerender(), names='value')
    stack_dd.observe(on_stack_change, names='value')
    slice_sl.observe(lambda _: _rerender(), names='value')
    show_gt_cb.observe(lambda _: _rerender(), names='value')

    # Initial load
    stacks = get_stacks(algo1_dd.value)
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        img0, *_ = _load(algo1_dd.value, stack_dd.value)
        slice_sl.max = img0.shape[0] - 1
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, 0, show_gt_cb.value)

    display(VBox([
        HBox([algo1_dd, algo2_dd, show_gt_cb]),
        HBox([stack_dd, slice_sl]),
        out,
    ]))